# EDA com Full Join das Abas

Este notebook carrega o Excel `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, faz um *full join* entre as abas `PEDE2022`, `PEDE2023`, `PEDE2024` pela chave `RA` e gera um EDA básico do resultado.

 INDE = Indice de desenvolvimento educacional uma media ponderada dos outros indicadores  
 IAN - Indicadores de adequação de nível{ (defasagem) em relação a fase atual}  
 IDA - Indicador de desenvolvimento acadêmico {exames de avaliação interna}  
 IEG - Indicador de Engajamento {Engajamento nas atividades curriculares ou voluntariado}  
 IAA - Indicador de auto avaliação {autoavaliação de sentimento}  
 IPS - Indicador Psicossocial {avaliação psico de interação social,emocional e comportamental}  
 IPP - Indicador Psicopedagógico {avaliação de psico no aprendizado, cognitivo}  
 IPV - Indicador do Ponto de Virada {avaliação psico de tipo de QI e engajamento}  

crianca 10 anos serie certa, indicadores relação de tipo essa crianca no fim do ano vai ter mais dificuldades 
+ dificuldade
+ atencao estudos

------
todas as criancas 5 serie,   
algumas estao atrasadas ( + idade ) = + defasagem
notas mais baixas média dos indicadores = + defasagem { depedendo do indicador ++defasagem ou + defasagem ou ++++ defasagem}

combinacoes
idade mt atrasada(++defasagem), notas ruins(+defasagem), engajamentoruim(+defasagem)
idade mt atrasada(++defasagem), notas boas(+defasagem), engajamento ruim(+defasagem)
idade atrasada(+defasagem), notas boas(+defasagem), engajamento ruim(+defasagem)
idade ok(+defasagem), notas ruim(+defasagem), engajamento +/- (+defasagem)


------------- ideia ------------
base de dados (input)
ano nascimento, serie, notas. (+ indicadores de impacto, se tiver indicador x previsão é mais assertiva)

output
nota de risco de defasagem, grau de risco (nota >7 muito alto, 3 > nota < 7 risco medio, nota <3 risco baixo)


In [1]:
import pandas as pd

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

FILE_PATH = "/workspaces/Datathon-Machine-Learning-Engineering/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# ============================================================================
# 1. CARREGAR DADOS
# ============================================================================

print("📂 Carregando dados...")
df_2022 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

print(f"  ✓ 2022: {df_2022.shape[0]:,} linhas x {df_2022.shape[1]} colunas")
print(f"  ✓ 2023: {df_2023.shape[0]:,} linhas x {df_2023.shape[1]} colunas")
print(f"  ✓ 2024: {df_2024.shape[0]:,} linhas x {df_2024.shape[1]} colunas")

📂 Carregando dados...
  ✓ 2022: 860 linhas x 42 colunas
  ✓ 2023: 1,014 linhas x 48 colunas
  ✓ 2024: 1,156 linhas x 50 colunas


In [2]:
COLUNAS_SEM_SUFIXO = ['RA']

# Função para adicionar sufixo nas colunas
def add_suffix_to_columns(df, suffix, exclude_cols):
    """
    Adiciona sufixo às colunas, exceto as que estão na lista de exclusão
    """
    new_columns = {}
    for col in df.columns:
        if col in exclude_cols:
            new_columns[col] = col  # Mantém o nome original
        else:
            new_columns[col] = f"{col}_{suffix}"  # Adiciona sufixo
    
    return df.rename(columns=new_columns)

# Renomeia cada base
df_2022_renamed = add_suffix_to_columns(df_2022, '2022', COLUNAS_SEM_SUFIXO)
df_2023_renamed = add_suffix_to_columns(df_2023, '2023', COLUNAS_SEM_SUFIXO)
df_2024_renamed = add_suffix_to_columns(df_2024, '2024', COLUNAS_SEM_SUFIXO)

print(df_2022_renamed.columns.tolist())
print(df_2023_renamed.columns.tolist())
print(df_2024_renamed.columns.tolist())

['RA', 'Fase_2022', 'Turma_2022', 'Nome_2022', 'Ano nasc_2022', 'Idade 22_2022', 'Gênero_2022', 'Ano ingresso_2022', 'Instituição de ensino_2022', 'Pedra 20_2022', 'Pedra 21_2022', 'Pedra 22_2022', 'INDE 22_2022', 'Cg_2022', 'Cf_2022', 'Ct_2022', 'Nº Av_2022', 'Avaliador1_2022', 'Rec Av1_2022', 'Avaliador2_2022', 'Rec Av2_2022', 'Avaliador3_2022', 'Rec Av3_2022', 'Avaliador4_2022', 'Rec Av4_2022', 'IAA_2022', 'IEG_2022', 'IPS_2022', 'Rec Psicologia_2022', 'IDA_2022', 'Matem_2022', 'Portug_2022', 'Inglês_2022', 'Indicado_2022', 'Atingiu PV_2022', 'IPV_2022', 'IAN_2022', 'Fase ideal_2022', 'Defas_2022', 'Destaque IEG_2022', 'Destaque IDA_2022', 'Destaque IPV_2022']
['RA', 'Fase_2023', 'INDE 2023_2023', 'Pedra 2023_2023', 'Turma_2023', 'Nome Anonimizado_2023', 'Data de Nasc_2023', 'Idade_2023', 'Gênero_2023', 'Ano ingresso_2023', 'Instituição de ensino_2023', 'Pedra 20_2023', 'Pedra 21_2023', 'Pedra 22_2023', 'Pedra 23_2023', 'INDE 22_2023', 'INDE 23_2023', 'Cg_2023', 'Cf_2023', 'Ct_2023'

In [3]:
# Padronizacao das fases (simples, coluna por coluna)
FASE_MAP = {
    "0": "ALFA",
    "1": "FASE 1",
    "2": "FASE 2",
    "3": "FASE 3",
    "4": "FASE 4",
    "5": "FASE 5",
    "6": "FASE 6",
    "7": "FASE 7",
    "8": "FASE 8",
    "9": "FASE 9",
}

# Fase 2022
fase_2022_base = df_2022_renamed["Fase_2022"].astype("string").str.strip().str.upper()
fase_2022_digit = fase_2022_base.str.extract(r"(\d)", expand=False)
df_2022_renamed["Fase_2022_adj"] = fase_2022_digit.map(FASE_MAP).fillna(fase_2022_base)

# Fase 2023 (ja correta, mas padroniza caixa/espacos)
fase_2023_base = df_2023_renamed["Fase_2023"].astype("string").str.strip().str.upper()
fase_2023_digit = fase_2023_base.str.extract(r"(\d)", expand=False)
df_2023_renamed["Fase_2023_adj"] = fase_2023_digit.map(FASE_MAP).fillna(fase_2023_base)

# Fase 2024 (mistura numero e texto -> pega o primeiro digito)
fase_2024_base = df_2024_renamed["Fase_2024"].astype("string").str.strip().str.upper()
fase_2024_digit = fase_2024_base.str.extract(r"(\d)", expand=False)
df_2024_renamed["Fase_2024_adj"] = fase_2024_digit.map(FASE_MAP).fillna(fase_2024_base)



In [4]:
print(sorted(map(int, df_2022_renamed["Defas_2022"].dropna().unique())))
print(sorted(map(int, df_2023_renamed["Defasagem_2023"].dropna().unique())))
print(sorted(map(int, df_2024_renamed["Defasagem_2024"].dropna().unique())))

[-5, -4, -3, -2, -1, 0, 1, 2]
[-4, -3, -2, -1, 0, 1, 2]
[-3, -2, -1, 0, 1, 2, 3]


In [5]:
# 4. CORRELAÇÃO COM DEFAS_2022 (PRÉ-MERGE, BASE 2022)

base = df_2022_renamed.copy()
target_col = "Defas_2022"

# Mantém apenas colunas numéricas (convertendo quando possível)
numeric_df = base.apply(lambda s: pd.to_numeric(s, errors="coerce"))

if target_col not in numeric_df.columns:
    raise ValueError(f"A coluna alvo {target_col} não está disponível para cálculo numérico.")

# Correlação de Spearman (mais robusta para relações monotônicas)
corr_target = numeric_df.corr(method="spearman")[target_col].dropna()
corr_target = corr_target.drop(index=target_col, errors="ignore")

top_corr = (
    corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)
    .head(20)
    .to_frame(name="corr_spearman")
    .reset_index()
    .rename(columns={"index": "variavel"})
)

print("\n📊 Top variáveis mais correlacionadas com", target_col)
display(top_corr)


📊 Top variáveis mais correlacionadas com Defas_2022


,variavel,corr_spearman
0,IAN_2022,0.886585
1,INDE 22_2022,0.416497
2,Cg_2022,-0.416471
3,Cf_2022,-0.388717
4,Idade 22_2022,-0.326028
5,Ano nasc_2022,0.326028
6,Ct_2022,-0.323271
7,Inglês_2022,0.232317
8,IEG_2022,0.199860
9,IPV_2022,0.153828


In [6]:
# 5. MÉDIAS POR FASE COM TODAS AS VARIÁVEIS (BASE 2022)
base_2022 = df_2022_renamed.copy()

# Colunas a excluir do cálculo de média
excluir = ["RA", "Fase_2022", "Fase_2022_adj"]

# Converte para numérico quando possível
for col in base_2022.columns:
    if col not in excluir:
        base_2022[col] = pd.to_numeric(base_2022[col], errors="coerce")

# Variáveis numéricas válidas
cols_numericas = [
    c for c in base_2022.columns
    if c not in excluir and base_2022[c].notna().any() and pd.api.types.is_numeric_dtype(base_2022[c])
]

# Apenas média por fase
medias_por_fase_todas = (
    base_2022.groupby("Fase_2022_adj")[cols_numericas]
    .mean()
    .round(2)
    .reset_index()
    .sort_values("Fase_2022_adj")
)

print(f"Total de variáveis numéricas consideradas: {len(cols_numericas)}")
display(medias_por_fase_todas)

Total de variáveis numéricas consideradas: 18


,Fase_2022_adj,Ano nasc_2022,Idade 22_2022,Ano ingresso_2022,INDE 22_2022,Cg_2022,Cf_2022,Ct_2022,Nº Av_2022,IAA_2022,IEG_2022,IPS_2022,IDA_2022,Matem_2022,Portug_2022,Inglês_2022,IPV_2022,IAN_2022,Defas_2022
0,ALFA,2013.07,8.93,2021.66,7.37,348.13,95.5,5.12,2.00,8.98,8.09,7.01,7.14,7.40,6.86,NaN,7.56,6.80,-0.92
1,FASE 1,2011.37,10.63,2020.80,7.20,392.58,96.5,6.78,2.76,8.64,8.52,7.08,6.46,5.91,6.99,NaN,7.36,5.74,-1.07
2,FASE 2,2009.91,12.09,2020.33,6.96,464.43,78.0,7.21,3.00,8.41,8.17,6.82,5.41,4.67,6.12,NaN,7.34,6.40,-0.83
3,FASE 3,2008.20,13.80,2020.27,6.60,520.86,74.5,7.22,3.91,7.49,7.07,6.72,5.14,4.91,5.63,5.13,6.55,6.99,-0.91
4,FASE 4,2007.07,14.93,2019.05,7.01,439.32,38.5,7.29,4.00,7.71,7.66,6.61,6.05,5.38,6.49,6.31,7.21,6.48,-0.96
5,FASE 5,2005.95,16.05,2019.47,6.88,472.83,30.5,6.05,3.55,8.11,7.34,6.86,5.87,5.54,5.81,6.42,7.26,6.25,-1.05
6,FASE 6,2005.17,16.83,2019.22,7.20,374.78,9.5,9.50,4.00,6.51,7.03,7.96,6.69,7.81,4.41,7.86,8.22,5.83,-0.83
7,FASE 7,2003.67,18.33,2019.33,6.64,530.71,11.0,6.05,4.00,7.01,7.24,6.52,5.25,5.58,4.09,5.90,7.18,6.19,-0.76


In [10]:

# Primeiro merge: 2022 + 2023
merged = df_2022_renamed.merge(
    df_2023_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)


# Segundo merge: (2022+2023) + 2024
merged = merged.merge(
    df_2024_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)

print(merged.shape)


(1661, 141)


In [11]:
total_alunos = merged['RA'].nunique()
print(f"  • Total de alunos únicos (RA): {total_alunos:,}")

# Alunos por ano
alunos_2022 = df_2022_renamed['RA'].nunique()
alunos_2023 = df_2023_renamed['RA'].nunique()
alunos_2024 = df_2024_renamed['RA'].nunique()

print(f"\n  • Alunos em 2022: {alunos_2022:,}")
print(f"  • Alunos em 2023: {alunos_2023:,}")
print(f"  • Alunos em 2024: {alunos_2024:,}")

# Alunos que aparecem em todos os anos
alunos_2022_2023 = set(df_2022_renamed['RA']) & set(df_2023_renamed['RA'])
alunos_2023_2024 = set(df_2023_renamed['RA']) & set(df_2024_renamed['RA'])
alunos_todos_anos = alunos_2022_2023 & set(df_2024_renamed['RA'])

print(f"\n  • Alunos em 2022 E 2023: {len(alunos_2022_2023):,}")
print(f"  • Alunos em 2023 E 2024: {len(alunos_2023_2024):,}")
print(f"  • Alunos todos anos: {len(alunos_todos_anos):,}")


  • Total de alunos únicos (RA): 1,661

  • Alunos em 2022: 860
  • Alunos em 2023: 1,014
  • Alunos em 2024: 1,156

  • Alunos em 2022 E 2023: 600
  • Alunos em 2023 E 2024: 765
  • Alunos todos anos: 468


In [12]:
print(merged.head())

print("\n📊 Info do dataset:")
print(f"  • Shape: {merged.shape}")
print(f"  • Colunas: {merged.shape[1]}")
print(f"  • Memória: {merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

        RA  Fase_2022 Turma_2022  Nome_2022  Ano nasc_2022  Idade 22_2022  \
0     RA-1        7.0          A    Aluno-1         2003.0           19.0   
1    RA-10        7.0          A   Aluno-10         2004.0           18.0   
2   RA-100        4.0          A  Aluno-100         2009.0           13.0   
3  RA-1000        NaN        NaN        NaN            NaN            NaN   
4  RA-1001        NaN        NaN        NaN            NaN            NaN   

  Gênero_2022  Ano ingresso_2022 Instituição de ensino_2022 Pedra 20_2022  \
0      Menina             2016.0             Escola Pública      Ametista   
1      Menina             2021.0             Escola Pública           NaN   
2      Menina             2019.0               Rede Decisão      Ametista   
3         NaN                NaN                        NaN           NaN   
4         NaN                NaN                        NaN           NaN   

   ... IAN_2024          Fase Ideal_2024  Defasagem_2024  Destaque IEG_202

In [14]:
# Tabela de fase x serie ideal x idade x ano nascimento (ano atual)

from datetime import datetime

ano_ref = datetime.now().year

rows = [
    {"FASE": "ALFA", "FASE_SERIE": "ALFA (1° e 2° ano)", "SERIE": "1° ano", "IDADE": 7},
    {"FASE": "ALFA", "FASE_SERIE": "ALFA (1° e 2° ano)", "SERIE": "2° ano", "IDADE": 8},
    {"FASE": "FASE 1", "FASE_SERIE": "Fase 1 (3° e 4° ano)", "SERIE": "3° ano", "IDADE": 9},
    {"FASE": "FASE 1", "FASE_SERIE": "Fase 1 (3° e 4° ano)", "SERIE": "4° ano", "IDADE": 10},
    {"FASE": "FASE 2", "FASE_SERIE": "Fase 2 (5° e 6° ano)", "SERIE": "5° ano", "IDADE": 11},
    {"FASE": "FASE 2", "FASE_SERIE": "Fase 2 (5° e 6° ano)", "SERIE": "6° ano", "IDADE": 12},
    {"FASE": "FASE 3", "FASE_SERIE": "Fase 3 (7° e 8° ano)", "SERIE": "7° ano", "IDADE": 13},
    {"FASE": "FASE 3", "FASE_SERIE": "Fase 3 (7° e 8° ano)", "SERIE": "8° ano", "IDADE": 14},
    {"FASE": "FASE 4", "FASE_SERIE": "Fase 4 (9° ano)", "SERIE": "9° ano", "IDADE": 15},
    {"FASE": "FASE 5", "FASE_SERIE": "Fase 5 (1° EM)", "SERIE": "1° EM", "IDADE": 16},
    {"FASE": "FASE 6", "FASE_SERIE": "Fase 6 (2° EM)", "SERIE": "2° EM", "IDADE": 17},
    {"FASE": "FASE 7", "FASE_SERIE": "Fase 7 (3° EM)", "SERIE": "3° EM", "IDADE": 18},
    {"FASE": "FASE 8", "FASE_SERIE": "Fase 8 (Universitários)", "SERIE": "Universidade", "IDADE": "18+"},
]

fase_serie_ideal = pd.DataFrame(rows)

fase_serie_ideal["ANO NASCIMENTO"] = fase_serie_ideal["IDADE"].apply(
    lambda idade: f"{ano_ref - 18}-" if idade == "18+" else str(ano_ref - int(idade))
)

fase_serie_ideal["IDADE"] = fase_serie_ideal["IDADE"].astype(str)

fase_serie_ideal = fase_serie_ideal[[
    "FASE",
    "FASE_SERIE",
    "SERIE",
    "IDADE",
    "ANO NASCIMENTO",
]]

display(fase_serie_ideal)


,FASE,FASE_SERIE,SERIE,IDADE,ANO NASCIMENTO
0,ALFA,ALFA (1° e 2° ano),1° ano,7,2019
1,ALFA,ALFA (1° e 2° ano),2° ano,8,2018
2,FASE 1,Fase 1 (3° e 4° ano),3° ano,9,2017
3,FASE 1,Fase 1 (3° e 4° ano),4° ano,10,2016
4,FASE 2,Fase 2 (5° e 6° ano),5° ano,11,2015
5,FASE 2,Fase 2 (5° e 6° ano),6° ano,12,2014
6,FASE 3,Fase 3 (7° e 8° ano),7° ano,13,2013
7,FASE 3,Fase 3 (7° e 8° ano),8° ano,14,2012
8,FASE 4,Fase 4 (9° ano),9° ano,15,2011
9,FASE 5,Fase 5 (1° EM),1° EM,16,2010
